In [2]:
import pandas as pd
import re
from textblob import TextBlob

# Load the dataset
df = pd.read_csv("reviews.csv",delimiter='\t')
total_rows=df.shape[0]

/var/folders/1s/tjfy4dhs42l6k94ly24b_4gw0000gn/T/ipykernel_41777/1084517343.py:6: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("reviews.csv",delimiter='\t')


In [3]:
# 1. Remove unnecessary columns
columns_to_keep = ['customer_id', 'review_id', 'product_id', 'product_title', 
                   'product_category', 'star_rating', 'helpful_votes', 
                   'total_votes', 'review_body', 'verified_purchase', 
                   'review_date', 'IP Address']
df = df[columns_to_keep]

In [9]:
star_1_rows = df[df['star_rating'] == 1.0].shape[0]
star_2_rows = df[df['star_rating'] == 2.0].shape[0]
star_3_rows = df[df['star_rating'] == 3.0].shape[0]
star_4_rows = df[df['star_rating'] == 4.0].shape[0]
star_5_rows = df[df['star_rating'] == 5.0].shape[0]
star_null = df[df['star_rating'] == ''].shape[0]
print(f"Number of rows where 'star_rating' is 1: {star_1_rows}")
print(f"Number of rows where 'star_rating' is 2: {star_2_rows}")
print(f"Number of rows where 'star_rating' is 3: {star_3_rows}")
print(f"Number of rows where 'star_rating' is 4: {star_4_rows}")
print(f"Number of rows where 'star_rating' is 5: {star_5_rows}")

print('negatives',star_1_rows+star_2_rows)
print('positives',star_4_rows+star_5_rows)
print('null',star_null)

Number of rows where 'star_rating' is 1: 5754
Number of rows where 'star_rating' is 2: 2108
Number of rows where 'star_rating' is 3: 2851
Number of rows where 'star_rating' is 4: 4876
Number of rows where 'star_rating' is 5: 17179
negatives 7862
positives 22055


In [54]:
df['genuine'] = True
# Initialize the 'failed_constraints' column as an empty list for each row
df['failed_constraints'] = [[] for _ in range(len(df))]

In [55]:
# 2. Flag rows where purchase is not verified
df.loc[df['verified_purchase'] != 'Y', 'failed_constraints'] = \
    df.loc[df['verified_purchase'] != 'Y', 'failed_constraints'].apply(lambda x: x + ['Unverified Purchase'])


In [56]:
# 3. Flag multiple reviews from the same IP for the same product
df['duplicate_review'] = df.duplicated(subset=['product_id', 'IP Address'], keep=False)
df.loc[df['duplicate_review'], 'failed_constraints'] = \
    df.loc[df['duplicate_review'], 'failed_constraints'].apply(lambda x: x + ['Multiple Reviews from Same IP'])


In [51]:
# 4. Check the purchase category and review relevance
def check_relevance(review_text, product_category):
    category_keywords = {
        "electronics": ["battery", "screen", "charger"],
        "books": ["story", "author", "pages"],
        "clothing": ["size", "fabric", "fit"],
    }
    keywords = category_keywords.get(product_category.lower(), [])
    return any(keyword in review_text.lower() for keyword in keywords)

df['relevant'] = df.apply(lambda x: check_relevance(str(x['review_body']), str(x['product_category'])), axis=1)
df.loc[~df['relevant'], 'failed_constraints'] = \
    df.loc[~df['relevant'], 'failed_constraints'].apply(lambda x: x + ['Irrelevant Review'])


In [ ]:
# 5. Check for profanity or spam
def contains_profanity(review_text):
    profanity_list = ['badword1', 'badword2']  # Add more profane words
    return any(profane_word in review_text.lower() for profane_word in profanity_list)

df['profanity'] = df['review_body'].apply(contains_profanity)
df.loc[df['profanity'], 'failed_constraints'] = \
    df.loc[df['profanity'], 'failed_constraints'].apply(lambda x: x + ['Contains Profanity or spam'])

#SPAM/Profanity
import re
from profanity_check import predict

# Helper function to detect URLs
def contains_url(text):
    url_pattern = r"http[s]?://|www\."
    return bool(re.search(url_pattern, text))

# Heuristic labeling function
def label_review(review):
    # Check for Spam
    spam_keywords = ["click here", "buy now", "free", "limited offer"]
    if contains_url(review) or any(kw in review.lower() for kw in spam_keywords):
        return "Spam"

    # Check for Offensive
    if predict([review])[0] == 1:  # Profanity-check returns 1 if offensive
        return "Offensive"

    # Check for Irrelevant
    if len(review.split()) < 3:  # Flag reviews with fewer than 3 words
        return "Irrelevant"

    # Otherwise, Legitimate
    return "Legitimate"


In [57]:
# 6. Match sentiment of star rating and review text
df['review_body'] = df['review_body'].fillna('').astype(str)  # Ensure all reviews are strings
df['star_rating'] = pd.to_numeric(df['star_rating'], errors='coerce')  # Ensure star_rating is numeric

def sentiment_matches(star_rating, review_text):
    try:
        sentiment = TextBlob(review_text).sentiment.polarity
        if star_rating >= 4 and sentiment > 0.2:  # Positive rating
            return True
        elif star_rating <= 2 and sentiment < -0.2:  # Negative rating
            return True
        elif star_rating == 3 and -0.2 <= sentiment <= 0.2:  # Neutral rating
            return True
    except:
        pass
    return False

df['sentiment_match'] = df.apply(lambda x: sentiment_matches(x['star_rating'], x['review_body']), axis=1)
df.loc[~df['sentiment_match'], 'failed_constraints'] = \
    df.loc[~df['sentiment_match'], 'failed_constraints'].apply(lambda x: x + ['Sentiment Mismatch'])


In [58]:
# Combine all failed constraints into a single 'genuine' flag
df['genuine'] = df['failed_constraints'].apply(lambda x: len(x) == 0)

In [59]:
# 7. Sort by helpfulness score (Optional: no impact on 'genuine' flag)
df['helpfulness_score'] = df['helpful_votes'] / (df['total_votes'] + 1)  # Avoid division by zero
# Sort by genuine reviews first, then by helpfulness score
df = df.sort_values(by=['genuine', 'helpfulness_score'], ascending=[False, False])

In [60]:
# Save the updated dataset
df.to_csv("flagged_reviews.csv", index=False)

print("Dataset updated with 'genuine' flag and saved as 'flagged_reviews.csv'.")

Dataset updated with 'genuine' flag and saved as 'flagged_reviews.csv'.


In [61]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
import pandas as pd


In [62]:
# Assuming your data is in a DataFrame called 'df'
df=pd.read_csv('flagged_reviews.csv')
# Assuming 'text_column' is the column containing your text data
df = df[df['review_body'].notna()]

In [63]:
from sklearn.preprocessing import LabelEncoder

columns_to_encode = ['verified_purchase', 'genuine','sentiment_match','duplicate_review']

# Initialize LabelEncoder
encoder = LabelEncoder()

# Apply LabelEncoder on specified columns
for col in columns_to_encode:
    df[col] = encoder.fit_transform(df[col])

# Show the encoded DataFrame
df.head()


,customer_id,review_id,product_id,product_title,product_category,star_rating,helpful_votes,total_votes,review_body,verified_purchase,review_date,IP Address,genuine,failed_constraints,duplicate_review,sentiment_match,helpfulness_score
0,52448148,RJAI8B34I8LKP,B001VIGCMK,"GPX Portable CD Player with AM/FM Radio, Line ...",Mobile_Electronics,5.0,100.0,100.0,This is a very nice cd player for the price (I...,1,2010-03-15,202.190.154.9,1,[],0,1,0.990099
1,14416835,R1I7E6NSDYHFU4,B001CEKDVO,CrazyOnDigital Premium Black Leather Case Appl...,Mobile_Electronics,5.0,84.0,84.0,I ordered the Apple iPod case after I had seen...,1,2011-04-20,192.37.185.123,1,[],0,1,0.988235
2,27325189,R2956RXM8GBHM0,B004N85WJE,RadTech 13-860 ProCable Shortz Dock Extender f...,Mobile_Electronics,5.0,65.0,65.0,I am so happy the dock extendor came early. I ...,1,2011-04-18,220.194.91.65,1,[],0,1,0.984848
3,39124978,R36C4VJMSPVQ23,B004A83PE6,6-Pack Mirror Screen Protector for Apple iPod ...,Mobile_Electronics,5.0,44.0,44.0,I brought this to replace the other mirror scr...,1,2011-02-05,207.14.198.211,1,[],0,1,0.977778
4,53096553,R1VJ340UMPD6N5,B0011XV19O,M-241.: Holux M-241 Bluetooth Data Logger GPS ...,Mobile_Electronics,4.0,44.0,44.0,"This is a good compact GPS logger, which is pr...",1,2008-11-02,196.207.179.205,1,[],0,1,0.977778


In [65]:
# Assuming your DataFrame is named 'df' and the column you're interested in is 'genuine'
print(f"Total number of rows: {total_rows}")
# Count where 'genuine' is 0

count_genuine_0 = df[df['genuine'] == 0].shape[0]
print(f"Number of rows where 'genuine' is 0: {count_genuine_0}")

# Count where 'genuine' is 1
count_genuine_1 = df[df['genuine'] == 1].shape[0]
print(f"Number of rows where 'genuine' is 1: {count_genuine_1}")



Total number of rows: 88578
Number of rows where 'genuine' is 0: 47267
Number of rows where 'genuine' is 1: 41296


In [15]:
# Assuming your DataFrame is named 'df'
genuine_0_rows = df[df['genuine'] == 0]
genuine_0_rows


,customer_id,review_id,product_id,product_title,product_category,star_rating,helpful_votes,total_votes,review_body,verified_purchase,review_date,IP Address,genuine,failed_constraints,duplicate_review,sentiment_match,helpfulness_score
41296,35579476,RXLG4F0S9T8UU,B000QJFOTM,Holux M-1000 32 Channel Wireless Bluetooth GPS...,Mobile_Electronics,5.0,112.0,112.0,"This was my first Amazon product review, thank...",0,2007-08-29,206.220.61.151,0,"['Unverified Purchase', 'Sentiment Mismatch']",0,0,0.991150
41297,29962857,R306TEXNYIF17X,B000P80S9K,6 in 1 Car Kit for Ipod (Black) Car Charger / ...,Mobile_Electronics,4.0,109.0,109.0,I love this product overall. But can only give...,1,2007-12-30,217.212.183.122,0,['Sentiment Mismatch'],0,0,0.990909
41298,37857121,RFF73VFTYNY38,B000EEGZ8S,Sudoku Puzzle With Modernized Touch Screen PDA,Mobile_Electronics,1.0,87.0,87.0,I just received this in the mail today and can...,0,2006-08-05,212.31.181.82,0,"['Unverified Purchase', 'Sentiment Mismatch']",0,0,0.988636
41299,42651247,R203WXG4O3HOJP,B0042SNZNK,Black Genuine Leather Case/Cover With Adjustab...,Mobile_Electronics,5.0,425.0,429.0,I pre-ordered this case for my Kindle 3 becaus...,1,2010-10-27,201.41.140.167,0,['Sentiment Mismatch'],0,0,0.988372
41300,52303174,R2VWCL4AM0T1O0,B001ESQSY4,(6 Colors available) Travel Package for Amazon...,Mobile_Electronics,5.0,77.0,77.0,I know several reviewers have been negative ab...,1,2008-10-29,200.195.219.229,0,['Sentiment Mismatch'],0,0,0.987179
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88560,51895028,RW4DAC97PHLBS,B00006I5UX,SYS TECHNOLOGY 1006 Portable CD MP3 Player,Mobile_Electronics,4.0,0.0,1.0,I do not own and have never tried this idem. ...,0,2002-10-23,200.188.108.4,0,['Unverified Purchase'],0,1,0.000000
88561,34643434,R33ZZ4VXWIZJVS,B00005OTZQ,Royal SE 2800 Hand-Held Spot Cleaner,Mobile_Electronics,1.0,0.0,0.0,This machine has broken down on me 4 times dur...,0,2002-09-21,206.185.78.109,0,"['Unverified Purchase', 'Sentiment Mismatch']",0,0,0.000000
88562,34643434,R1LNQR1JD2NKN7,B00005OTZQ,Royal SE 2800 Hand-Held Spot Cleaner,Mobile_Electronics,1.0,0.0,0.0,"I received this as a gift, and have had it ser...",0,2002-09-21,214.181.136.68,0,"['Unverified Purchase', 'Sentiment Mismatch']",0,0,0.000000
88563,35427145,R1FYGTQEVATD3F,B00005OTZQ,Royal SE 2800 Hand-Held Spot Cleaner,Mobile_Electronics,5.0,0.0,1.0,This works great! I have one child who happens...,0,2002-08-27,196.38.116.137,0,['Unverified Purchase'],0,1,0.000000


In [16]:


# Split the dataset into features (X) and target (y)
X = df.drop(columns=['genuine','failed_constraints','duplicate_review','sentiment_match','helpfulness_score'], axis=1)  # Features (excluding 'genuine' column)
y = df['genuine']  # Target variable

# Split data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocess the 'review_body' text data (TF-IDF)
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)  # Limit to top 1000 features
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train['review_body'])
X_test_tfidf = tfidf_vectorizer.transform(X_test['review_body'])

# Optionally, combine the numerical features with the text data (for simplicity, assume all other columns are numerical)
X_train_numerical = X_train.drop(['review_body', 'product_title','customer_id','review_id','product_id','product_category','IP Address','review_date'], axis=1)  # Remove non-numerical columns
X_test_numerical = X_test.drop(['review_body', 'product_title','customer_id','review_id','product_id','product_category','IP Address','review_date'], axis=1)  # Remove non-numerical columns

# Scale numerical features if needed
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_numerical)
X_test_scaled = scaler.transform(X_test_numerical)

# Combine TF-IDF and numerical features (we'll stack them horizontally)
X_train_combined = hstack([X_train_scaled, X_train_tfidf])
X_test_combined = hstack([X_test_scaled, X_test_tfidf])


In [17]:
# Define a list of models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Support Vector Machine': SVC()
}

# Create a dictionary to store results
results = {}

# Train and evaluate each model
for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train the model
    model.fit(X_train_combined, y_train)

    # Make predictions on the test set
    y_pred = model.predict(X_test_combined)

    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    
    # Store the results
    results[model_name] = accuracy

    # Print classification report
    print(f"{model_name} Accuracy: {accuracy:.4f}")
    print(classification_report(y_test, y_pred))
    print("-" * 60)

# Display the accuracy scores for all models
print("\nModel Comparison (Accuracy Scores):")
for model_name, accuracy in results.items():
    print(f"{model_name}: {accuracy:.4f}")


Training Logistic Regression...
Logistic Regression Accuracy: 0.8198
              precision    recall  f1-score   support

           0       0.83      0.83      0.83      9415
           1       0.80      0.81      0.81      8298

    accuracy                           0.82     17713
   macro avg       0.82      0.82      0.82     17713
weighted avg       0.82      0.82      0.82     17713

------------------------------------------------------------
Training Random Forest...
Random Forest Accuracy: 0.8524
              precision    recall  f1-score   support

           0       0.86      0.86      0.86      9415
           1       0.84      0.84      0.84      8298

    accuracy                           0.85     17713
   macro avg       0.85      0.85      0.85     17713
weighted avg       0.85      0.85      0.85     17713

------------------------------------------------------------
Training Gradient Boosting...
Gradient Boosting Accuracy: 0.8068
              precision    recall

from sklearn.model_selection import GridSearchCV

# Example: Tuning a Random Forest Classifier
rf_model = RandomForestClassifier(random_state=42)
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
}

grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_combined, y_train)

# Best parameters found by GridSearchCV
print("Best parameters:", grid_search.best_params_)

# Evaluate the tuned model
y_pred_tuned = grid_search.predict(X_test_combined)
print(f"Tuned Random Forest Accuracy: {accuracy_score(y_test, y_pred_tuned):.4f}")
